In [ ]:
import pandas as pd
import plotly.graph_objects as go

In [ ]:
def load_impacto_mensal(proposta:str)->pd.DataFrame:
    fname = f'impacto_mensal_{proposta}.csv'
    return pd.read_csv(fname, index_col=0, sep=';')

In [ ]:
impactos_mensais ={
    'Equiparação com a tabela dos AMCI' : load_impacto_mensal('amci'),
    'Reajuste pelo IPC-Fipe - jun. 2016' : load_impacto_mensal('ipc'),
    #'Reajuste pelo IPC-Fipe - jun. 2021' : load_impacto_mensal('ipc_nunes'),
    "Situação atual" : load_impacto_mensal('amci')[['nivel_carreira', 'valor_total_prefeitura_atual']].rename({'valor_total_prefeitura_atual': 'valor_total_prefeitura_proposta'}, axis=1)
}

In [ ]:
impactos_mensais['Equiparação com a tabela dos AMCI']

In [ ]:

def gerar_grafico_niveis_df(df, col_valor:str, titulo:str, nome_arquivo="grafico_niveis.png"):
    
    col_nivel = 'nivel_carreira'
    
    cores = ["steelblue"] * len(df)
    indice_maximo = df[col_valor].idxmax()
    cores[indice_maximo] = "crimson"

    # Formatação para o texto sobre as barras (R$ 1.234,56)
    texto_formatado = df[col_valor].apply(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    fig = go.Figure(data=[
        go.Bar(
            x=df[col_nivel],
            y=df[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        # Define os separadores globalmente: o primeiro é o decimal, o segundo é o de milhar
        separators=",.",
        plot_bgcolor="white",
        width=1200,
        height=600,
        xaxis=dict(
            title="Níveis de Carreira",
            tickangle=-45,
            categoryorder="array",
            categoryarray=df[col_nivel].tolist()
        ),
        yaxis=dict(
            title="Valores em R$",
            showgrid=True,
            gridcolor="lightgrey",
            # Formata os números do eixo Y com separador de milhar e 2 casas decimais
            tickformat=",2f"
        ),
        margin=dict(l=50, r=50, t=100, b=120)
    )

    fig.write_image(nome_arquivo)

    return fig

In [ ]:
for nome, df in impactos_mensais.items():
    titulo = f"Custo total por nível: {nome}"
    nome_arquivo = f"grafico_niveis_{nome.replace(' ', '_').lower()}.png"
    gerar_grafico_niveis_df(df, "valor_total_prefeitura_proposta", titulo, nome_arquivo)

In [ ]:
for proposta, df in impactos_mensais.items():
    valor_total = df['valor_total_prefeitura_proposta'].sum()*12
    valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    print(f"Valor total anual: {valor_formatado} - {proposta}")

In [ ]:
for proposta, df in impactos_mensais.items():
    try:
        valor_total = df['impacto_mensal'].sum()*12
        valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        print(f"Impacot total anual: {valor_formatado} - {proposta}")
    except KeyError:
        print(proposta, "não possui impacto")

In [ ]:
niveis = pd.read_csv('tabelas_vencimentos_utilizadas.csv', sep=';', index_col=0)

In [ ]:
niveis = niveis[niveis['nome_tabela']!='original_atualizada_ipc_nunes'].reset_index(drop=True)

In [ ]:
niveis

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def exportar_graficos_comparativos(df, col_valor="vencimento"):
    col_nivel = "nivel"
    col_tabela = "nome_tabela"
    
    # Identifica as tabelas que serão comparadas com a 'atual'
    tabelas_extras = [t for t in df[col_tabela].unique() if t != "atual"]
    
    for tabela in tabelas_extras:
        # Filtra apenas o par necessário
        df_par = df[df[col_tabela].isin(["atual", tabela])]
        
        fig = go.Figure()

        # Adiciona as barras para 'atual' e para a 'tabela' da vez
        for nome in ["atual", tabela]:
            df_sub = df_par[df_par[col_tabela] == nome]
            
            # Formatação Real (sem centavos para evitar sobreposição de texto)
            texto = df_sub[col_valor].apply(
                lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
            )

            fig.add_trace(go.Bar(
                x=df_sub[col_nivel],
                y=df_sub[col_valor],
                name=nome.replace("_", " ").title(),
                text=texto,
                textposition="outside",
                marker_color="steelblue" if nome == "atual" else "crimson"
            ))

        fig.update_layout(
            title=dict(
                text=f"Comparativo: Atual vs {tabela.replace('_', ' ').title()}",
                x=0.5,
                font=dict(size=22)
            ),
            barmode="group",
            separators=",.",
            plot_bgcolor="white",
            width=1200,
            height=600,
            xaxis=dict(title="Nível de Carreira", type="category"),
            yaxis=dict(
                title="Vencimento (R$)",
                showgrid=True,
                gridcolor="lightgrey",
                tickformat=",0f"
            ),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(l=50, r=50, t=100, b=50)
        )

        # Salva cada gráfico com o nome da tabela correspondente
        nome_arquivo = f"comparativo_atual_vs_{tabela}.png"
        fig.write_image(nome_arquivo, engine="kaleido")
        print(f"Arquivo salvo: {nome_arquivo}")

# Exemplo de uso:
# exportar_graficos_comparativos(df)

In [ ]:
exportar_graficos_comparativos(niveis)

In [ ]:
impacto = pd.read_csv('impactos_anuais.csv', sep=';', index_col=0)

In [ ]:
impacto

In [ ]:
impacto = impacto.drop(0)

In [ ]:
impacto= impacto.rename({'nivel' : 'Proposta Reajusta', 'vencimento' : "Impacto orçamentário anualizado"}, axis=1)

In [ ]:
impacto

In [ ]:
impacto['Proposta Reajusta'] = impacto['Proposta Reajusta'].str.replace('_', ' ').str.replace('ipca', 'icp fipe').str.title()

In [ ]:
impacto

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def gerar_grafico_simples_ordenado(df, col_valor="vencimento", titulo="Comparativo de Reajustes", nome_arquivo="grafico_simples.png"):
    
    col_tabela = "Proposta Reajusta"
    
    # Ordena o DataFrame pelo valor do vencimento (menor para o maior)
    df_ordenado = df.sort_values(by=col_valor, ascending=True)

    # Formatação para o texto sobre as barras (R$ 1.234)
    texto_formatado = df_ordenado[col_valor].apply(
        lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    # Define cores: Azul para a 'atual' e Steelblue para as demais
    cores = ["steelblue" if n != "atual" else "darkblue" for n in df_ordenado[col_tabela]]

    fig = go.Figure(data=[
        go.Bar(
            x=df_ordenado[col_tabela],
            y=df_ordenado[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            font=dict(size=22)
        ),
        separators=",.",
        plot_bgcolor="white",
        width=1000,
        height=600,
        xaxis=dict(
            title="Cenários / Tabelas",
            tickangle=0 # Mantém reto para facilitar leitura se forem poucos nomes
        ),
        yaxis=dict(
            title="Vencimento (R$)",
            showgrid=True,
            gridcolor="lightgrey",
            tickformat=",0f"
        ),
        margin=dict(l=50, r=50, t=100, b=100)
    )

    fig.write_image(nome_arquivo, engine="kaleido")

    return fig

In [ ]:
gerar_grafico_simples_ordenado(impacto, col_valor='Impacto orçamentário anualizado')

In [ ]:
situacao_atual = pd.read_csv('situacao_atual.csv', sep=';', index_col=0)

In [ ]:
situacao_atual.head()

In [ ]:
situacao_atual['nome'].str.startswith('recem_nomeado').sum()

In [ ]:
qtd_por_nivel = situacao_atual.groupby('nivel_carreira').count()[['rf']].rename({'rf' : 'Quantidade'}, axis=1)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(
        x=qtd_por_nivel.index.astype(str),  # Convertendo o índice para string para melhor exibição
        y=qtd_por_nivel['Quantidade'],
        text=qtd_por_nivel['Quantidade'],
        textposition='outside',
        marker_color='steelblue'
    )
])

fig.update_layout(
    title=dict(
        text="Quantidade por Nível na Carreira",
        x=0.5,
        font=dict(size=22)
    ),
    xaxis=dict(
        title="Nível na Carreira",
        tickangle=0
    ),
    yaxis=dict(
        title="Quantidade",
        showgrid=True,
        gridcolor="lightgrey"
    ),
    plot_bgcolor="white",
    width=800,
    height=500,
    margin=dict(l=50, r=50, t=100, b=50)
)
fig.write_image('quantidade_pessoas_por_nivel.png', engine="kaleido")

fig.show()